## 1. Create Widget

In [0]:
import json

# INPUT

raw_metadata = dbutils.widgets.get("table_metadata")

if not raw_metadata:
    raise ValueError("table_metadata is required")

table_metadata = json.loads(raw_metadata)

if isinstance(table_metadata, list):
    table_metadata = table_metadata[0]

# EXTRACT (GENERIC)

table_id = int(table_metadata["table_id"])
table_name = table_metadata["table_name"]

source_system = table_metadata["source_system"]
source_schema = table_metadata["source_schema"]
source_table = table_metadata["source_table"]
source_path = table_metadata["source_path"]

bronze_schema = table_metadata["bronze_schema"]
silver_schema = table_metadata["silver_schema"]

print(f"Loaded parameters for table: {table_name}")

## 2. Read Table Parameters

In [0]:
params_df = (
    spark.table("banking.metadata.table_parameters")
         .filter(f"table_id = {table_id}")
)

## 3. Convert to Single JSON Object

In [0]:
rows = params_df.select("parameter_name", "parameter_value").collect()

parameters_dict = {
    row.parameter_name: row.parameter_value
    for row in rows
}

print("Parameters JSON Object:")
print(parameters_dict)

## 4. Set Databricks Task Value

In [0]:
dbutils.jobs.taskValues.set(
    key="table_parameters",
    value=parameters_dict
)

print("Task value 'table_parameters' has been set.")